# TCGA-BRCA Preprocessing Pipeline
### Three prediction targets:
- `label_survival` — vital status (Dead=1, Alive=0)
- `label_recurrence` — progression or recurrence (Yes=1, No=0)
- `label_treatment_response` — complete response (Yes=1, else=0)

### Pipeline:
```
Metadata JSON ──► file_id → patient_id map
                                          ├──► RNA-seq TPM matrix (patients × genes)
                                          │         ↓
Clinical TSVs ──► Merge ──► Labels ──► Feature Engineering
                                          ↓
                              Inner join on patient_id
                                          ↓
                         X_combined.csv + y_labels.csv
```

## 0. Imports & Paths

In [1]:
import os, glob, json, pickle, warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
warnings.filterwarnings('ignore')

CLINICAL_BASE = '/kaggle/input/datasets/jannatulferdoushritu/clinical'
GENOME_BASE   = '/kaggle/input/datasets/jannatulferdoushritu/genomefixed'
META_PATH     = '/kaggle/input/datasets/jannatulferdoushritu/metadata/metadata.cart.2026-05-06.json'
OUT           = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

rna_files = glob.glob(GENOME_BASE + '/**/*.tsv', recursive=True)
print(f'Clinical exists : {os.path.exists(CLINICAL_BASE)}')
print(f'Metadata exists : {os.path.exists(META_PATH)}')
print(f'RNA-seq files   : {len(rna_files)}')
print('OK')

Clinical exists : True
Metadata exists : True
RNA-seq files   : 4472
OK


## 1. Build file_id → patient_id Map from Metadata JSON
**Key fix:** use `associated_entities` not `cases` — confirmed from your metadata structure.

In [2]:
with open(META_PATH) as f:
    meta = json.load(f)

print(f'Metadata entries: {len(meta)}')

records = []
for entry in meta:
    file_id   = entry.get('file_id', '')
    file_name = entry.get('file_name', '')
    entities  = entry.get('associated_entities', [])
    for e in entities:
        # entity_submitter_id is the TCGA barcode e.g. TCGA-E2-A1IU-01A-...
        eid = e.get('entity_submitter_id', '')
        # Trim to patient barcode (first 3 parts: TCGA-XX-XXXX)
        patient_id = '-'.join(eid.split('-')[:3]) if eid else ''
        records.append({
            'file_id'   : file_id,
            'file_name' : file_name,
            'patient_id': patient_id
        })

meta_map = pd.DataFrame(records).drop_duplicates('file_id')
meta_map = meta_map[meta_map['patient_id'] != '']
print(f'file_id → patient_id mappings: {len(meta_map)}')
print(meta_map.head(5).to_string())

Metadata entries: 1231
file_id → patient_id mappings: 1231
                                file_id                                                                    file_name    patient_id
0  744a6d3d-b666-49aa-8d26-47f34e3d1eb5  94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.augmented_star_gene_counts.tsv  TCGA-BH-A18H
1  4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5  df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.augmented_star_gene_counts.tsv  TCGA-E2-A14P
2  1ace2a0c-773d-45b5-8fd6-968c88731bbb  c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.augmented_star_gene_counts.tsv  TCGA-AN-A04A
3  2d5b0962-b5c4-4482-9f28-47e4dcdb6df6  602e3d24-a45d-4fe4-9b70-a679d93f470d.rna_seq.augmented_star_gene_counts.tsv  TCGA-5L-AAT1
4  84cff855-d17a-4c09-b432-53893ae69c1e  03d891b3-8faf-4384-94ce-2015f1ca5df0.rna_seq.augmented_star_gene_counts.tsv  TCGA-GI-A2C8


## 2. Index RNA-seq Files → patient_id
Each RNA-seq file lives in a folder named after its file_id (UUID).

In [3]:
# Build index: filepath → file_id (folder name) → patient_id
file_index = pd.DataFrame({
    'filepath': rna_files,
    'file_id' : [os.path.basename(os.path.dirname(p)) for p in rna_files]
})

file_index = file_index.merge(meta_map[['file_id','patient_id']], on='file_id', how='left')

matched   = file_index['patient_id'].notna().sum()
unmatched = file_index['patient_id'].isna().sum()
print(f'Total TSV files : {len(file_index)}')
print(f'Matched to patient : {matched}')
print(f'Unmatched (log files etc): {unmatched}')

# Keep only matched files
file_index = file_index.dropna(subset=['patient_id']).reset_index(drop=True)
# Keep one file per patient (drop duplicates)
file_index = file_index.drop_duplicates('patient_id').reset_index(drop=True)
print(f'Unique patients with RNA-seq: {len(file_index)}')

Total TSV files : 4472
Matched to patient : 4472
Unmatched (log files etc): 0
Unique patients with RNA-seq: 1095


## 3. Load All RNA-seq TPM Files → patients × genes matrix
- Extract `tpm_unstranded` column only
- Skip N_ summary rows
- Index by ENSG gene_id (version stripped)

In [4]:
def load_tpm(filepath):
    """Load tpm_unstranded from one STAR augmented TSV.
    Returns pd.Series(gene_id → tpm_value)."""
    df = pd.read_csv(filepath, sep='\t', comment='#',
                     usecols=['gene_id','tpm_unstranded'],
                     low_memory=False)
    # Drop STAR summary rows
    df = df[~df['gene_id'].astype(str).str.startswith('N_')]
    # Strip gene version: ENSG00000000003.15 → ENSG00000000003
    df['gene_id'] = df['gene_id'].astype(str).str.split('.').str[0]
    df['tpm_unstranded'] = pd.to_numeric(df['tpm_unstranded'], errors='coerce').fillna(0.0)
    return df.set_index('gene_id')['tpm_unstranded']

# Load all files efficiently
tpm_dict = {}
failed   = []

for _, row in tqdm(file_index.iterrows(), total=len(file_index), desc='Loading RNA-seq'):
    try:
        tpm_dict[row['patient_id']] = load_tpm(row['filepath'])
    except Exception as e:
        failed.append((row['patient_id'], str(e)))

print(f'Loaded  : {len(tpm_dict)} patients')
print(f'Failed  : {len(failed)}')

# Stack into matrix: patients (rows) × genes (cols)
genomic_raw = pd.DataFrame(tpm_dict).T
genomic_raw.index.name = 'patient_id'
print(f'Raw genomic matrix: {genomic_raw.shape}')

Loading RNA-seq: 100%|██████████| 1095/1095 [03:08<00:00,  5.82it/s]


Loaded  : 1095 patients
Failed  : 0
Raw genomic matrix: (1095, 60660)


## 4. Normalize Genomic Matrix
Standard RNA-seq pipeline:
1. Variance filter (remove near-zero genes)
2. log1p (stabilize variance)
3. Top-2000 most variable genes (by std)
4. Per-gene z-score

In [5]:
print(f'Before: {genomic_raw.shape}')

# 1. Variance filter
sel = VarianceThreshold(threshold=0.1)
gm  = sel.fit_transform(genomic_raw.fillna(0))
kept_genes = genomic_raw.columns[sel.get_support()]
gm_df = pd.DataFrame(gm, index=genomic_raw.index, columns=kept_genes)
print(f'After variance filter : {gm_df.shape[1]} genes')

# 2. log1p
gm_df = np.log1p(gm_df)

# 3. Top-2000 most variable genes
N_TOP = min(2000, gm_df.shape[1])
top_genes = gm_df.std().nlargest(N_TOP).index
gm_df = gm_df[top_genes].copy()
print(f'After top-{N_TOP} MAD   : {gm_df.shape[1]} genes')

# 4. Z-score
gscaler = StandardScaler()
gm_scaled = gscaler.fit_transform(gm_df)
genomic_matrix = pd.DataFrame(gm_scaled, index=gm_df.index, columns=top_genes)
print(f'After z-score         : mean={genomic_matrix.values.mean():.4f}  std={genomic_matrix.values.std():.4f}')
print(f'Final genomic matrix  : {genomic_matrix.shape}')
print('OK')

Before: (1095, 60660)
After variance filter : 33730 genes
After top-2000 MAD   : 2000 genes
After z-score         : mean=-0.0000  std=1.0000
Final genomic matrix  : (1095, 2000)
OK


## 5. PCA on Genomic Matrix (2000 genes → 100 PCs)

In [6]:
N_PCA = 100
pca = PCA(n_components=N_PCA, random_state=42)
genomic_pca = pd.DataFrame(
    pca.fit_transform(genomic_matrix),
    index=genomic_matrix.index,
    columns=[f'PC{i+1}' for i in range(N_PCA)]
)
print(f'PCA shape          : {genomic_pca.shape}')
print(f'Variance explained : {pca.explained_variance_ratio_.sum():.1%}')

PCA shape          : (1095, 100)
Variance explained : 73.4%


## 6. Load & Merge Clinical Files

In [7]:
SENTINEL = "'--"

def load_tsv(fname):
    df = pd.read_csv(os.path.join(CLINICAL_BASE, fname), sep='\t', low_memory=False)
    df = df.replace(SENTINEL, np.nan)
    # Drop GDC annotation row if present
    if df.shape[0] > 0 and str(df.iloc[0,0]).lower().startswith('field'):
        df = df.iloc[1:].reset_index(drop=True)
    return df

clin = load_tsv('clinical.tsv')
fu   = load_tsv('follow_up.tsv')
path = load_tsv('pathology_detail.tsv')
expo = load_tsv('exposure.tsv')
# family_history.tsv is empty — skip

PAT = 'cases.submitter_id'  # confirmed patient ID column

# Filter BRCA only
for df in [clin, fu, path, expo]:
    df.rename(columns={PAT: 'patient_id'}, inplace=True)

clin = clin[clin['project.project_id']=='TCGA-BRCA'].copy()
fu   = fu[fu['project.project_id']=='TCGA-BRCA'].copy()
path = path[path['project.project_id']=='TCGA-BRCA'].copy()

# One row per patient in clinical
clin = clin.drop_duplicates('patient_id', keep='first').set_index('patient_id')

# Latest follow-up per patient
fu_day = 'follow_ups.days_to_follow_up'
if fu_day in fu.columns:
    fu[fu_day] = pd.to_numeric(fu[fu_day], errors='coerce')
    fu = fu.sort_values(fu_day, ascending=False)
fu_agg   = fu.drop_duplicates('patient_id', keep='first').set_index('patient_id')
fu_agg.columns   = ['fu.'   + c for c in fu_agg.columns]

path_agg = path.drop_duplicates('patient_id', keep='first').set_index('patient_id')
path_agg.columns = ['path.' + c for c in path_agg.columns]

expo_agg = expo.drop_duplicates('patient_id', keep='first').set_index('patient_id')
expo_agg.columns = ['expo.' + c for c in expo_agg.columns]

merged = clin.join(fu_agg, how='left').join(path_agg, how='left').join(expo_agg, how='left')
print(f'Clinical patients : {len(clin)}')
print(f'Merged shape      : {merged.shape}')

Clinical patients : 1098
Merged shape      : (1098, 471)


## 7. Engineer 3 Prediction Labels

In [8]:
df = merged.copy()

def find_col(df, *kws):
    for kw in kws:
        hits = [c for c in df.columns if kw.lower() in c.lower()]
        if hits: return hits[0]
    return None

# Label 1: Survival
vital = find_col(df, 'vital_status')
if vital:
    df['label_survival'] = (df[vital].astype(str).str.lower()=='dead').astype(int)
    print(f'label_survival       <- {vital}')
    print(f'  Dead={df["label_survival"].sum()}  Alive={(df["label_survival"]==0).sum()}')
else:
    df['label_survival'] = np.nan
    print('label_survival: vital_status not found')

# Label 2: Recurrence
recur = find_col(df, 'progression_or_recurrence')
drec  = find_col(df, 'days_to_recurrence')
if recur:
    flag = df[recur].astype(str).str.lower().isin(['yes','1','true'])
    if drec:
        flag = flag | (pd.to_numeric(df[drec], errors='coerce') > 0).fillna(False)
    df['label_recurrence'] = flag.astype(int)
    print(f'\nlabel_recurrence     <- {recur}')
    print(f'  Yes={df["label_recurrence"].sum()}  No={(df["label_recurrence"]==0).sum()}')
else:
    df['label_recurrence'] = np.nan
    print('label_recurrence: column not found')

# Label 3: Treatment Response
resp = find_col(df, 'treatment_outcome')
if resp:
    df['label_treatment_response'] = (
        df[resp].astype(str).str.lower().str.contains('complete response', na=False)
    ).astype(int)
    print(f'\nlabel_treatment_response <- {resp}')
    print(f'  Complete={df["label_treatment_response"].sum()}  Other={(df["label_treatment_response"]==0).sum()}')
else:
    df['label_treatment_response'] = np.nan
    print('label_treatment_response: column not found')

LABEL_COLS = ['label_survival','label_recurrence','label_treatment_response']
print(f'\nLabel non-null counts:')
print(df[LABEL_COLS].notna().sum())

label_survival       <- demographic.vital_status
  Dead=152  Alive=946

label_recurrence     <- diagnoses.progression_or_recurrence
  Yes=0  No=1098

label_treatment_response <- treatments.treatment_outcome
  Complete=125  Other=973

Label non-null counts:
label_survival              1098
label_recurrence            1098
label_treatment_response    1098
dtype: int64


## 8. Clinical Feature Engineering

In [9]:
feat_parts = []

# ── Numeric ───────────────────────────────────────────────────────────────────
NUM = [
    'demographic.age_at_index',
    'demographic.days_to_birth',
    'demographic.days_to_death',
    'diagnoses.age_at_diagnosis',
    'diagnoses.days_to_last_follow_up',
    'diagnoses.days_to_diagnosis',
    'fu.follow_ups.days_to_follow_up',
    'fu.follow_ups.days_to_recurrence',
    'path.pathology_details.lymph_nodes_tested',
    'path.pathology_details.lymph_nodes_positive_by_he',
    'path.pathology_details.tumor_largest_dimension_diameter',
    'expo.exposures.bmi',
    'expo.exposures.cigarettes_per_day',
    'expo.exposures.years_smoked',
]
num_cols = [c for c in NUM if c in df.columns]
if num_cols:
    num_df = df[num_cols].apply(pd.to_numeric, errors='coerce')
    num_df = num_df.fillna(num_df.median())
    scaler_clin = StandardScaler()
    feat_parts.append(pd.DataFrame(
        scaler_clin.fit_transform(num_df),
        index=num_df.index, columns=num_cols
    ))
    print(f'Numeric  : {len(num_cols)} features')

# ── Ordinal ───────────────────────────────────────────────────────────────────
ORDINAL = {
    'diagnoses.ajcc_pathologic_stage': [
        'not reported','stage i','stage ia','stage ib','stage ii',
        'stage iia','stage iib','stage iii','stage iiia','stage iiib','stage iiic','stage iv'],
    'diagnoses.ajcc_pathologic_t': [
        'not reported','tx','t0','tis','t1','t1a','t1b','t1c',
        't2','t2a','t2b','t3','t4','t4a','t4b','t4c','t4d'],
    'diagnoses.ajcc_pathologic_n': [
        'not reported','nx','n0','n0 (i-)','n0 (i+)','n0 (mol+)',
        'n1','n1a','n1b','n1c','n1mi','n2','n2a','n2b','n3','n3a','n3b','n3c'],
    'diagnoses.ajcc_pathologic_m': ['not reported','mx','m0','m1'],
    'diagnoses.tumor_grade'       : ['not reported','g1','g2','g3','g4'],
}
ord_cols = {k:v for k,v in ORDINAL.items() if k in df.columns}
if ord_cols:
    ord_df = df[list(ord_cols.keys())].fillna('not reported').astype(str)
    ord_df = ord_df.apply(lambda c: c.str.lower().str.strip())
    cats = []
    for col in ord_df.columns:
        base   = ord_cols[col]
        unseen = list(set(ord_df[col].unique()) - set(base))
        cats.append(base + unseen)
    enc = OrdinalEncoder(categories=cats, handle_unknown='use_encoded_value', unknown_value=-1)
    feat_parts.append(pd.DataFrame(
        enc.fit_transform(ord_df),
        index=ord_df.index,
        columns=[f'ord_{c.split(".")[-1]}' for c in ord_df.columns]
    ))
    print(f'Ordinal  : {len(ord_cols)} features')

# ── Nominal (one-hot) ─────────────────────────────────────────────────────────
NOM = [
    'demographic.gender','demographic.race','demographic.ethnicity',
    'diagnoses.primary_diagnosis','diagnoses.morphology',
    'diagnoses.tissue_or_organ_of_origin',
    'fu.follow_ups.disease_response',
    'fu.follow_ups.progression_or_recurrence',
    'path.pathology_details.margin_status',
    'path.pathology_details.breast_carcinoma_estrogen_receptor_status',
    'path.pathology_details.breast_carcinoma_progesterone_receptor_status',
    'path.pathology_details.her2_immunohistochemistry_level_result',
    'expo.exposures.alcohol_history',
]
nom_cols = [c for c in NOM if c in df.columns]
if nom_cols:
    nom_df = df[nom_cols].fillna('unknown').astype(str)
    nom_df = nom_df.apply(lambda c: c.str.lower().str.strip())
    nom_df.columns = [c.split('.')[-1] for c in nom_cols]
    ohe = pd.get_dummies(nom_df, drop_first=False)
    feat_parts.append(ohe)
    print(f'Nominal  : {ohe.shape[1]} one-hot cols from {len(nom_cols)} features')

# ── Lymph node ratio (derived feature) ───────────────────────────────────────
pc = 'path.pathology_details.lymph_nodes_positive_by_he'
tc = 'path.pathology_details.lymph_nodes_tested'
if pc in df.columns and tc in df.columns:
    pos  = pd.to_numeric(df[pc], errors='coerce').fillna(0)
    test = pd.to_numeric(df[tc], errors='coerce').fillna(0)
    ln   = (pos / test.replace(0, np.nan)).fillna(0).clip(0,1)
    feat_parts[0]['ln_ratio'] = ln.values
    print(f'LN ratio : added (mean={ln.mean():.3f})')

clin_feat = pd.concat(feat_parts, axis=1)
print(f'\nClinical feature matrix: {clin_feat.shape}')

Numeric  : 11 features
Ordinal  : 5 features
Nominal  : 103 one-hot cols from 10 features

Clinical feature matrix: (1098, 119)


## 9. Intersect Clinical + Genomic → Final Cohort

In [10]:
clin_pts    = set(clin_feat.index)
genomic_pts = set(genomic_matrix.index)
both = sorted(clin_pts & genomic_pts)

print(f'Clinical patients    : {len(clin_pts)}')
print(f'Genomic patients     : {len(genomic_pts)}')
print(f'Clinical only        : {len(clin_pts - genomic_pts)}')
print(f'Genomic only         : {len(genomic_pts - clin_pts)}')
print(f'BOTH modalities      : {len(both)}  ← final cohort')

clin_both    = clin_feat.loc[both]
genomic_both = genomic_matrix.loc[both]
pca_both     = genomic_pca.loc[both]
labels       = df.loc[both, LABEL_COLS]

print(f'\nLabel counts in cohort:')
print(labels.notna().sum())

Clinical patients    : 1098
Genomic patients     : 1095
Clinical only        : 3
Genomic only         : 0
BOTH modalities      : 1095  ← final cohort

Label counts in cohort:
label_survival              1095
label_recurrence            1095
label_treatment_response    1095
dtype: int64


## 10. Build Final Feature Matrices

In [11]:
# Option A: Clinical + Full genomic (2000 genes)
X_combined = clin_both.join(genomic_both, how='inner')

# Option B: Clinical + PCA genomic (100 PCs) — lighter for MLP/SVM
X_pca = clin_both.join(pca_both, how='inner')

y = labels.loc[X_combined.index]

print('Feature matrices:')
print(f'  X_combined (clin+2000genes) : {X_combined.shape}')
print(f'  X_pca      (clin+100PCs)    : {X_pca.shape}')
print(f'  X_clinical  only            : {clin_both.shape}')
print(f'  X_genomic   only            : {genomic_both.shape}')
print(f'\ny labels shape: {y.shape}')
print('\nLabel distributions:')
for lbl in LABEL_COLS:
    n  = y[lbl].notna().sum()
    pr = y[lbl].mean() if n > 0 else float('nan')
    print(f'  {lbl:<38} n={n:>4}  positive_rate={pr:.1%}')

Feature matrices:
  X_combined (clin+2000genes) : (1095, 2119)
  X_pca      (clin+100PCs)    : (1095, 219)
  X_clinical  only            : (1095, 119)
  X_genomic   only            : (1095, 2000)

y labels shape: (1095, 3)

Label distributions:
  label_survival                         n=1095  positive_rate=13.9%
  label_recurrence                       n=1095  positive_rate=0.0%
  label_treatment_response               n=1095  positive_rate=11.3%


## 11. Quality Report

In [12]:
print('='*55)
print('QUALITY REPORT')
print('='*55)
print(f'Final cohort           : {len(X_combined)} patients')
print(f'Total features         : {X_combined.shape[1]}')
print(f'  Clinical             : {clin_both.shape[1]}')
print(f'  Genomic (full)       : {genomic_both.shape[1]}')
print(f'NaN rate in X          : {X_combined.isna().mean().mean():.2%}')
print()
print('Label summary:')
for lbl in LABEL_COLS:
    n  = y[lbl].notna().sum()
    pr = y[lbl].mean() if n > 0 else float('nan')
    print(f'  {lbl:<38} n={n:>4}  pos_rate={pr:.1%}')

QUALITY REPORT
Final cohort           : 1095 patients
Total features         : 2119
  Clinical             : 119
  Genomic (full)       : 2000
NaN rate in X          : 0.09%

Label summary:
  label_survival                         n=1095  pos_rate=13.9%
  label_recurrence                       n=1095  pos_rate=0.0%
  label_treatment_response               n=1095  pos_rate=11.3%


## 12. Save Outputs

In [13]:
# Save all feature matrices
X_combined.to_csv(os.path.join(OUT, 'X_combined.csv'))
X_pca.to_csv(os.path.join(OUT, 'X_pca.csv'))
clin_both.to_csv(os.path.join(OUT, 'X_clinical.csv'))
genomic_both.to_csv(os.path.join(OUT, 'X_genomic.csv'))
y.to_csv(os.path.join(OUT, 'y_labels.csv'))

# Patient list for histopathology teammate
pd.Series(both, name='patient_id').to_csv(
    os.path.join(OUT, 'cohort_patient_ids.csv'), index=False
)

# Save preprocessors for reproducibility
with open(os.path.join(OUT, 'preprocessors.pkl'), 'wb') as f:
    pickle.dump({
        'variance_selector': sel,
        'genomic_scaler'   : gscaler,
        'pca'              : pca,
        'top_genes'        : top_genes.tolist(),
        'label_cols'       : LABEL_COLS,
    }, f)

print('Saved:')
for fname in ['X_combined.csv','X_pca.csv','X_clinical.csv',
              'X_genomic.csv','y_labels.csv',
              'cohort_patient_ids.csv','preprocessors.pkl']:
    fp = os.path.join(OUT, fname)
    if os.path.exists(fp):
        print(f'  {fname:<35} {os.path.getsize(fp)/1024/1024:.1f} MB')

print('\nPipeline complete!')
print('Share cohort_patient_ids.csv with your histopathology teammate.')
print('They filter WSI embeddings to these patient IDs, then you join on patient_id.')

Saved:
  X_combined.csv                      41.9 MB
  X_pca.csv                           2.9 MB
  X_clinical.csv                      0.8 MB
  X_genomic.csv                       41.1 MB
  y_labels.csv                        0.0 MB
  cohort_patient_ids.csv              0.0 MB
  preprocessors.pkl                   3.1 MB

Pipeline complete!
Share cohort_patient_ids.csv with your histopathology teammate.
They filter WSI embeddings to these patient IDs, then you join on patient_id.


In [14]:
import pandas as pd
import numpy as np

fu_raw = pd.read_csv('/kaggle/input/datasets/jannatulferdoushritu/clinical/follow_up.tsv', sep='\t')

# Check all columns related to recurrence/progression
recur_cols = [c for c in fu_raw.columns if any(kw in c.lower() for kw in 
              ['recur', 'progress', 'disease', 'response', 'event', 'status'])]

print(f'Recurrence-related columns found: {len(recur_cols)}')
for col in recur_cols:
    vc = fu_raw[col].value_counts(dropna=False)
    print(f'\n{col}:')
    print(vc.head(6).to_string())

Recurrence-related columns found: 22

follow_ups.adverse_event:
follow_ups.adverse_event
'--    9427

follow_ups.adverse_event_grade:
follow_ups.adverse_event_grade
'--    9427

follow_ups.cause_of_response:
follow_ups.cause_of_response
'--    9427

follow_ups.days_to_adverse_event:
follow_ups.days_to_adverse_event
'--    9427

follow_ups.days_to_first_event:
follow_ups.days_to_first_event
'--    9427

follow_ups.days_to_progression:
follow_ups.days_to_progression
'--     9340
301        4
1351       2
443        2
524        2
192        2

follow_ups.days_to_progression_free:
follow_ups.days_to_progression_free
'--    9427

follow_ups.days_to_recurrence:
follow_ups.days_to_recurrence
'--    9398
596       2
136       1
181       1
155       1
301       1

follow_ups.disease_response:
follow_ups.disease_response
'--              6397
TF-Tumor Free    2684
WT-With Tumor     288
Unknown            58

follow_ups.ecog_performance_status:
follow_ups.ecog_performance_status
'--    9427

fo

In [15]:
# ── Step 1: Reload raw follow_up with all columns ─────────────────────────────
SENTINEL = "'--"

fu_raw = pd.read_csv(
    '/kaggle/input/datasets/jannatulferdoushritu/clinical/follow_up.tsv',
    sep='\t', low_memory=False
)
fu_raw = fu_raw.replace(SENTINEL, np.nan)
fu_raw = fu_raw.rename(columns={'cases.submitter_id': 'patient_id'})
fu_raw = fu_raw[fu_raw['project.project_id'] == 'TCGA-BRCA']

# Latest follow-up per patient
fu_raw['follow_ups.days_to_follow_up'] = pd.to_numeric(
    fu_raw['follow_ups.days_to_follow_up'], errors='coerce'
)
fu_raw = fu_raw.sort_values('follow_ups.days_to_follow_up', ascending=False)
fu_latest = fu_raw.drop_duplicates('patient_id', keep='first').set_index('patient_id')

# ── Step 2: Fix recurrence label ──────────────────────────────────────────────
sig1 = fu_latest['follow_ups.progression_or_recurrence'].astype(str).str.lower() == 'yes'
sig2 = pd.to_numeric(fu_latest['follow_ups.days_to_recurrence'],  errors='coerce') > 0
sig3 = pd.to_numeric(fu_latest['follow_ups.days_to_progression'], errors='coerce') > 0
sig4 = fu_latest['follow_ups.disease_response'].astype(str).str.lower().str.contains('with tumor', na=False)

recurrence_label = (sig1 | sig2.fillna(False) | sig3.fillna(False) | sig4).astype(int)
print('Recurrence label:')
print(recurrence_label.value_counts())
print(f'Positive rate: {recurrence_label.mean():.1%}')

# ── Step 3: Add extra useful columns to feature matrix ────────────────────────
# These are informative columns from follow_up we didn't include before
EXTRA_NOMINAL = [
    'follow_ups.disease_response',
    'follow_ups.progression_or_recurrence_type',
    'other_clinical_attributes.menopause_status',
]
EXTRA_NUMERIC = [
    'follow_ups.days_to_recurrence',
    'follow_ups.days_to_progression',
]

extra_parts = []

# Numeric extras
num_extra_cols = [c for c in EXTRA_NUMERIC if c in fu_latest.columns]
if num_extra_cols:
    ne = fu_latest[num_extra_cols].apply(pd.to_numeric, errors='coerce')
    ne = ne.fillna(ne.median())
    sc = StandardScaler()
    extra_parts.append(pd.DataFrame(
        sc.fit_transform(ne),
        index=ne.index, columns=[f'extra_{c.split(".")[-1]}' for c in num_extra_cols]
    ))
    print(f'\nExtra numeric cols added: {len(num_extra_cols)}')

# Nominal extras
nom_extra_cols = [c for c in EXTRA_NOMINAL if c in fu_latest.columns]
if nom_extra_cols:
    ne = fu_latest[nom_extra_cols].fillna('unknown').astype(str)
    ne = ne.apply(lambda c: c.str.lower().str.strip())
    ne.columns = [c.split('.')[-1] for c in nom_extra_cols]
    ohe_extra = pd.get_dummies(ne, drop_first=False)
    extra_parts.append(ohe_extra)
    print(f'Extra nominal cols added: {ohe_extra.shape[1]} one-hot columns')

# Combine extras
extra_feat = pd.concat(extra_parts, axis=1)
print(f'Extra features shape: {extra_feat.shape}')

# ── Step 4: Update X_combined with extra features ─────────────────────────────
# Only keep patients in our cohort
extra_cohort = extra_feat.reindex(both).fillna(0)

X_combined_v2 = pd.concat([X_combined, extra_cohort], axis=1)
X_pca_v2      = pd.concat([X_pca,      extra_cohort], axis=1)

# ── Step 5: Update y labels ───────────────────────────────────────────────────
y['label_recurrence'] = recurrence_label.reindex(both).fillna(0).astype(int)

# ── Step 6: Final report ──────────────────────────────────────────────────────
print('\n' + '='*55)
print('UPDATED QUALITY REPORT')
print('='*55)
print(f'Cohort size       : {len(both)} patients')
print(f'X_combined_v2     : {X_combined_v2.shape}')
print(f'X_pca_v2          : {X_pca_v2.shape}')
print('\nLabel distributions:')
for lbl in LABEL_COLS:
    n  = y[lbl].notna().sum()
    pr = y[lbl].mean() if n > 0 else 0.0
    print(f'  {lbl:<38} n={n:>4}  pos_rate={pr:.1%}')

# ── Step 7: Save updated files ────────────────────────────────────────────────
X_combined_v2.to_csv(os.path.join(OUT, 'X_combined_v2.csv'))
X_pca_v2.to_csv(os.path.join(OUT, 'X_pca_v2.csv'))
y.to_csv(os.path.join(OUT, 'y_labels_v2.csv'))
print('\nSaved: X_combined_v2.csv, X_pca_v2.csv, y_labels_v2.csv')

Recurrence label:
0    976
1    120
Name: count, dtype: int64
Positive rate: 10.9%

Extra numeric cols added: 2
Extra nominal cols added: 7 one-hot columns
Extra features shape: (1096, 9)

UPDATED QUALITY REPORT
Cohort size       : 1095 patients
X_combined_v2     : (1095, 2128)
X_pca_v2          : (1095, 228)

Label distributions:
  label_survival                         n=1095  pos_rate=13.9%
  label_recurrence                       n=1095  pos_rate=11.0%
  label_treatment_response               n=1095  pos_rate=11.3%

Saved: X_combined_v2.csv, X_pca_v2.csv, y_labels_v2.csv


In [16]:
# Already saved — just check they exist
import os
OUT = '/kaggle/working'
for f in os.listdir(OUT):
    print(f)

X_combined.csv
cohort_patient_ids.csv
y_labels.csv
X_pca_v2.csv
X_pca.csv
X_clinical.csv
X_combined_v2.csv
preprocessors.pkl
X_genomic.csv
y_labels_v2.csv
.virtual_documents
